<a href="https://colab.research.google.com/github/nashranoor98/hospital-readmission-prediction/blob/main/CaseStudy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hospital Readmission Prediction — Logistic Regression with L2 Regularization
Predicting 30-day hospital readmission using the UCI Diabetes 130-US Hospitals dataset.

## 1. Importing and Studying Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv('data/diabetic_data.csv')
data = data.replace('?', np.nan)
print('Dataset shape:', data.shape)
display(data.head())

In [ ]:
data.info()
display(data.describe(include='all').T.head(15))
print('\nMissing values (top 10):')
print(data.isnull().sum().sort_values(ascending=False).head(10))
print('\nDuplicate rows:', data.duplicated().sum())

## 2. EDA and Visualisation

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=data, x='readmitted', order=['NO','>30','<30'])
plt.title('Readmission Distribution')
plt.xlabel('Readmission Category')
plt.ylabel('Number of Patients')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['time_in_hospital'], bins=14, kde=True)
plt.title('Distribution of Time in Hospital')
plt.xlabel('Days in Hospital')
plt.show()

## 3. Preparing Dataset

In [ ]:
data['target_30day'] = (data['readmitted'] == '<30').astype(int)
drop_cols = ['encounter_id','patient_nbr','readmitted','target_30day']
X = data.drop(columns=drop_cols)
y = data['target_30day']
high_missing = X.columns[X.isna().mean() > 0.90]
X = X.drop(columns=high_missing)
print('Target counts:')
print(y.value_counts())
print('Features retained:', X.shape[1])

## 4. Splitting Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns
numeric_pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical_pipe = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num',numeric_pipe,numeric),('cat',categorical_pipe,categorical)])
print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 5. Training Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model = Pipeline([('preprocess',preprocess),('logreg',LogisticRegression(penalty='l2',C=1.0,max_iter=1000,solver='liblinear'))])
model.fit(X_train,y_train)
print('Model training completed.')

## 6. Prediction and Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay
y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.50).astype(int)
print('ROC-AUC Score:', round(roc_auc_score(y_test,y_prob),4))
print(classification_report(y_test,y_pred,digits=4))
print('Confusion Matrix:')
print(confusion_matrix(y_test,y_pred))
RocCurveDisplay.from_predictions(y_test,y_prob)
plt.title('ROC Curve - 30-Day Readmission')
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test,y_pred)
plt.title('Confusion Matrix')
plt.show()

## 7. False Negative vs False Positive Cost
A **false negative** is a patient who is actually readmitted within 30 days but is not flagged by the model. This can mean a missed opportunity for follow-up or preventive intervention.

A **false positive** is a patient flagged as high-risk who is not readmitted. This can increase care-team workload and resource use. In this case study, the decision threshold should therefore be selected with the relative clinical and operational costs in mind.

This notebook is an educational case study and is not a clinical decision-support system.